# Intuición de Bag of Words

**Duración estimada:** 35 minutos

---

## Objetivo

Las computadoras solamente entienden números, pero nosotros nos comunicamos con palabras. **Bag of Words** ("bolsa de palabras") es la técnica más simple y fundamental para resolver ese problema: convertir texto en una tabla de números.

En este cuaderno vas a:
- construir una tabla Bag of Words **a mano**, antes de tocar código;
- automatizar el mismo proceso con Python;
- reflexionar sobre qué información conserva esta representación y qué pierde;
- ver un ejemplo práctico (detección de spam) que muestra para qué sirve.

## Resultados de aprendizaje
Al final deberías poder:
- explicar qué representa cada fila y cada columna de una tabla Bag of Words;
- distinguir vocabulario, documento y frecuencia;
- justificar por qué Bag of Words es útil como primer paso;
- anticipar limitaciones que después resolveremos con TF-IDF.

## Relación con los cuadernos anteriores
En `003_spacy/006_preprocesamiento_y_representacion_texto.ipynb` viste cómo pasar de texto crudo a listas de tokens. Acá damos el siguiente paso: convertir esos tokens en números.

---

## 1. ¿Qué es Bag of Words? (La Bolsa de Palabras)

Imaginá que tenés una bolsa mágica:

1. **Tomás un texto** → "El perro come carne"
2. **Cortás cada palabra** → ["El", "perro", "come", "carne"]
3. **Las metés en la bolsa** y las agitás
4. **Contás cuántas hay de cada una**

### Resultado: un "recibo" con cantidades
- "el": 1 vez
- "perro": 1 vez
- "come": 1 vez
- "carne": 1 vez

**¡Eso es Bag of Words!** Convertimos texto en una lista de números.

---

## 2. Ejercicio manual: ¡hacelo vos!

Antes de usar la computadora, practiquemos a mano.

### Nuestros textos de ejemplo
1. "El perro ladra fuerte"
2. "El gato come pescado"
3. "El perro come carne"

### Paso 1: encontrá todas las palabras únicas
Escribí acá todas las palabras diferentes que encuentres:

_Respuesta: el, perro, ladra, fuerte, gato, come, pescado, carne_

### Paso 2: armá la tabla manualmente

| Texto | el | perro | ladra | fuerte | gato | come | pescado | carne |
|-------|----|----- |-------|--------|------|------|---------|-------|
| Texto 1 | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 0 |
| Texto 2 | 1 | 0 | 0 | 0 | 1 | 1 | 1 | 0 |
| Texto 3 | 1 | 1 | 0 | 0 | 0 | 1 | 0 | 1 |

¡Felicidades! Acabás de crear tu primera representación Bag of Words.

---

## 3. Ahora con Python: ¡automático!

Obviamente, hacer esto a mano con miles de textos sería imposible. Python lo hace automáticamente.

In [ ]:
# Importamos las herramientas que necesitamos
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Los mismos textos del ejercicio manual
textos = [
    "El perro ladra fuerte",
    "El gato come pescado",
    "El perro come carne"
]

print("Nuestros textos originales:")
for i, texto in enumerate(textos):
    print(f"Texto {i+1}: {texto}")

In [ ]:
# Creamos el "contador automático"
contador = CountVectorizer()

# Le damos nuestros textos para que cuente las palabras
matriz_numeros = contador.fit_transform(textos)

# Vemos qué palabras encontró
palabras_encontradas = contador.get_feature_names_out()
print("Palabras únicas encontradas:")
print(palabras_encontradas)

print(f"\nTotal de palabras únicas: {len(palabras_encontradas)}")

In [ ]:
# Convertimos a una tabla bonita para ver mejor
tabla_bow = pd.DataFrame(
    matriz_numeros.toarray(),
    columns=palabras_encontradas,
    index=["Texto 1", "Texto 2", "Texto 3"]
)

print("Nuestra tabla Bag of Words:")
print(tabla_bow)

---

## 4. Momento de reflexión

### Observá la tabla anterior y respondé:

1. **¿Qué textos son más parecidos?** (Pista: mirá qué columnas tienen números similares)
2. **¿Qué palabra aparece en todos los textos?**
3. **¿Qué palabras son únicas de cada texto?**

### La magia de los números
- **Texto 1 y 3** comparten "el" y "perro" → son más parecidos.
- **"el"** aparece en todos → es muy común (¡quizás no tan importante!).
- **"ladra", "gato", "pescado"** son únicas → definen mejor cada texto.

---

## 5. Ejemplo práctico: detector de spam

Ahora veamos cómo esto funciona para algo útil: ¡detectar correos spam!

In [ ]:
# Emails de ejemplo
emails = [
    "Reunión mañana a las 9am en la oficina",       # Normal
    "OFERTA ESPECIAL! COMPRE AHORA! DINERO FÁCIL!", # Spam
    "Hola mamá, cómo estás? Te extraño mucho",      # Normal
    "GANE MILLONES! CLICK AQUÍ URGENTE!",            # Spam
    "Adjunto el informe que me pediste ayer"         # Normal
]

# Las etiquetas reales (lo que queremos predecir)
etiquetas = ["Normal", "Spam", "Normal", "Spam", "Normal"]

print("Nuestros emails de ejemplo:")
for i, (email, tipo) in enumerate(zip(emails, etiquetas)):
    print(f"{i+1}. [{tipo}] {email}")

In [ ]:
# Aplicamos Bag of Words a los emails
contador_email = CountVectorizer()
matriz_emails = contador_email.fit_transform(emails)

# Creamos la tabla
palabras_emails = contador_email.get_feature_names_out()
tabla_emails = pd.DataFrame(
    matriz_emails.toarray(),
    columns=palabras_emails,
    index=[f"Email {i+1} ({tipo})" for i, tipo in enumerate(etiquetas)]
)

# Mostramos solo las columnas más interesantes para que sea legible
columnas_interesantes = ["oferta", "especial", "dinero", "gane", "reunión", "mamá", "informe"]

# Filtramos las que realmente existen en la tabla
columnas_existentes = []
for col in columnas_interesantes:
    if col in tabla_emails.columns:
        columnas_existentes.append(col)

print("Tabla Bag of Words de los emails (columnas seleccionadas):")
print(tabla_emails[columnas_existentes])

In [ ]:
# Busquemos pistas: ¿qué palabras sugieren spam y cuáles son normales?
print("Análisis de sentimientos:")

print("\nPalabras que sugieren SPAM:")
palabras_spam = []
for p in palabras_emails:
    if p in ["oferta", "especial", "dinero", "gane", "millones", "urgente", "click"]:
        palabras_spam.append(p)
print(palabras_spam)

print("\nPalabras que sugieren NORMAL:")
palabras_normales = []
for p in palabras_emails:
    if p in ["reunión", "informe", "mamá", "adjunto", "oficina", "ayer"]:
        palabras_normales.append(p)
print(palabras_normales)

# Mostramos la tabla con las palabras clave
columnas_clave = palabras_spam + palabras_normales
if columnas_clave:
    print("\nTabla con palabras clave:")
    print(tabla_emails[columnas_clave])

---

## 6. ¿Por qué funciona?

### La intuición
- **Emails normales** usan palabras como: "reunión", "informe", "mamá", "ayer".
- **Emails spam** usan palabras como: "oferta", "dinero", "gratis", "urgente".

### Con Bag of Words
1. Convertimos cada email en números.
2. La computadora aprende qué números se asocian con spam.
3. ¡Puede predecir nuevos emails!

### Es como enseñarle a un chico
- "Cuando veas muchas palabras como OFERTA, GRATIS, DINERO juntas → probablemente es spam."
- "Cuando veas palabras como reunión, informe, adjunto → probablemente es trabajo."

---

## 7. Limitaciones: ¿qué perdemos?

Bag of Words es poderoso, pero tiene limitaciones:

### Perdemos el orden
- "No me gusta nada" vs "Me gusta mucho"
- ¡Ambos tienen las mismas palabras, pero significados opuestos!

### Perdemos el contexto
- "El banco del parque" vs "El banco donde guardo mi plata"
- "Banco" significa cosas diferentes, pero BoW no lo sabe.

### Pero aun así funciona porque:
- Con miles de textos, los patrones emergen.
- Es rápido y simple.
- Es la base para técnicas más avanzadas.

---

## Actividad breve

1. Buscá dos palabras que aparezcan en más de un texto del ejemplo inicial.
2. Buscá una palabra que haga único a un solo texto.
3. Explicá qué podrías afirmar a partir de la tabla y qué no podrías afirmar todavía.

## Cierre

En el próximo cuaderno vas a mantener esta base, pero vas a comparar tres representaciones: Bag of Words, **TF-IDF** (que le da menos peso a las palabras muy comunes) y **n-gramas** (que capturan frases como "Buenos Aires" en lugar de palabras sueltas).